# Can EEG tell which word was read? Raw EEG, pretrained EEG models and fine-tuning

Every earlier analysis used ZuCo's precomputed word features (band power over the reading time, or fixation-locked window means), and none of them recovered which word was read beyond its length and frequency. This notebook gives the EEG every remaining chance on the same data:

| representation | what it is |
|---|---|
| `trt`, `frp` | the earlier features, on exactly the same readers and words (for comparison) |
| `eye_tracking` | reading times only (no EEG): what the eye tracker alone says about the word |
| `raw` | the raw EEG time course 0–800 ms after the word's first fixation (105 channels × 16 bins) |
| `cbramod` | **pretrained EEG foundation model CBraMod** (ICLR 2025, braindecode weights), frozen |
| `neurolm` | **pretrained NeuroLM-B** tokenizer, frozen (optional) |
| `cbramod_ft` | CBraMod **fine-tuned** to match each word's EEG to the word's embedding (+ `cbramod_ft_shuffled`, the same training with words shuffled across EEG: the control) |

Every representation goes through the same tests on unseen sentences, each with its controls (shuffled EEG, noise, word length/frequency/position):
* telling two words apart (2-vs-2), also on pairs **matched** for length, frequency and position;
* picking the word among the test vocabulary (retrieval) and predicting its embedding;
* **identifying which sentence was read** among held-out sentences of the same length;
* word properties (length, frequency, position, surprisal) and the sentiment of the decoded text.

A final table compares them all. Stages push their results to `saved_results/word_information_<stage>/`. Use an **A100** (stage 3 fine-tunes a model). Only the 9 ZuCo readers whose files contain per-fixation EEG can be used.

## Setup

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/MohammadJRanjbar/zuco-multimodal-sentiment.git'
BRANCH = 'feature/neurolm-eeg-probe'
REPO_DIR = '/content/zuco-multimodal-sentiment'
if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin && git -C $REPO_DIR checkout $BRANCH && git -C $REPO_DIR -c user.name=colab -c user.email=colab@users.noreply.github.com pull --rebase origin $BRANCH
%cd $REPO_DIR
!pip install -q -r requirements-wordinfo.txt
!git log --oneline -1

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
THESIS_ROOT = '/content/drive/MyDrive/Thesis'
ARTIFACTS = f'{THESIS_ROOT}/CachedArtifacts/zuco_multimodal_sentiment'
MAT_DIR = f'{THESIS_ROOT}/Data/zuco_og_raw'
LABELS_CSV = f'{THESIS_ROOT}/Data/zuco_sentiment_labels_task1_fixed.csv'
WORD_EEG_DIR = f'{ARTIFACTS}/word_eeg/TRT'
FRP_DIR = f'{ARTIFACTS}/word_eeg/FRP'
EPOCHS_DIR = f'{ARTIFACTS}/word_epochs'          # raw 1-s epochs per word (about 2 GB)
REPS_DIR = f'{ARTIFACTS}/word_representations'  # one word-EEG cache per representation
RESULTS = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment/word_information'
NEUROLM_CHECKPOINT_DIR = f'{ARTIFACTS}/models/NeuroLM'
os.environ['HF_HOME'] = f'{ARTIFACTS}/hf_cache'
os.makedirs(RESULTS, exist_ok=True)
DEVICE = 'cuda' if os.system('nvidia-smi > /dev/null 2>&1') == 0 else 'cpu'
for path in [MAT_DIR, LABELS_CSV, WORD_EEG_DIR, FRP_DIR]:
    print('ok ' if os.path.exists(path) else 'MISSING', path)
print('device:', DEVICE)

# GitHub push: the GITHUB_TOKEN secret (fine-grained, Contents: Read and write on your fork).
os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
GIT_NAME, GIT_EMAIL = 'mohammadranjbar', 'mjrkalhar@gmail.com'
token = os.environ['GITHUB_TOKEN']
check = subprocess.run(['git', 'push', '--dry-run', REPO_URL.replace('https://', f'https://x-access-token:{token}@'),
                        f'HEAD:refs/heads/push-check-{os.getpid()}'], capture_output=True, text=True)
message = (check.stdout + check.stderr).replace(token, '***')
print('push check: OK' if check.returncode == 0 else 'push check FAILED; results stay on Drive.\n' + message[-800:])

def push(stage, *sources):
    """Commit the small result files of one stage to saved_results/word_information_<stage>/ and push."""
    sources = [s for s in sources if os.path.exists(s)]
    if not sources:
        print('nothing to push for', stage); return
    result = subprocess.run(['python', 'scripts/save_results_to_github.py', '--name', f'word_information_{stage}',
                             '--source', *sources, '--push', '--author-name', GIT_NAME, '--author-email', GIT_EMAIL],
                            capture_output=True, text=True)
    print((result.stdout + result.stderr).replace(token, '***')[-1500:])

In [ ]:
# The raw ZuCo files are read sentence by sentence; a local copy is much faster than Drive.
LOCAL_MAT = '/content/zuco_og_raw'
!mkdir -p $LOCAL_MAT && rsync -a --include='results*_SR.mat' --exclude='*' "$MAT_DIR/" "$LOCAL_MAT/"
!python -m pytest -q tests/test_wordinfo.py tests/test_frp.py

## Stage 1 — raw EEG around every word (about 30–60 min, resumable)

For each word's first fixation: the sentence EEG preprocessed as for NeuroLM (0.1–75 Hz, 50 Hz notch, average reference, 200 Hz) and cut from −200 to +800 ms. The log lists each reader's number of epochs; the 3 readers without per-fixation EEG are skipped.

In [ ]:
!python scripts/extract_word_epochs.py --mat-dir "$LOCAL_MAT" --labels-csv "$LABELS_CSV" --out-dir "$EPOCHS_DIR"
OUT_1 = f'{RESULTS}/epochs'
os.makedirs(OUT_1, exist_ok=True)
!cp "$EPOCHS_DIR/word_epochs.json" "$OUT_1/"
push('epochs', OUT_1)

## Stage 2 — word representations (about 15–30 min)

`raw`, `eye_tracking`, the earlier `trt` and `frp` features restricted to the same words, and the frozen pretrained **CBraMod** (downloaded from Hugging Face, `braindecode/cbramod-pretrained`).

In [ ]:
!python scripts/build_word_representations.py --epochs-dir "$EPOCHS_DIR" --out-root "$REPS_DIR" \
    --representations trt frp eye_tracking raw cbramod --trt-dir "$WORD_EEG_DIR" --frp-dir "$FRP_DIR" --device $DEVICE
OUT_2 = f'{RESULTS}/representations'
os.makedirs(OUT_2, exist_ok=True)
!cp "$REPS_DIR/representations.json" "$OUT_2/"
push('representations', OUT_2)

### Optional — frozen NeuroLM-B (the model of the earlier sentiment probe)

Uses the NeuroLM checkpoint already on Drive (downloaded once from `Weibang/NeuroLM` if missing). Set `RUN_NEUROLM = False` to skip.

In [ ]:
RUN_NEUROLM = True
if RUN_NEUROLM:
    import yaml
    from huggingface_hub import hf_hub_download
    NEUROLM_DIR = '/content/NeuroLM'
    if not os.path.exists(NEUROLM_DIR):
        !git clone -q https://github.com/935963004/NeuroLM.git $NEUROLM_DIR
    !git -C $NEUROLM_DIR checkout -q 0cda987
    ckpt = yaml.safe_load(open('configs/neurolm_probe.yaml'))['checkpoint']
    NEUROLM_CHECKPOINT = os.path.join(NEUROLM_CHECKPOINT_DIR, ckpt['filename'])
    if not os.path.exists(NEUROLM_CHECKPOINT):
        NEUROLM_CHECKPOINT = hf_hub_download(ckpt['repo'], ckpt['filename'], revision=ckpt['revision'],
                                             local_dir=NEUROLM_CHECKPOINT_DIR)
    !python scripts/build_word_representations.py --epochs-dir "$EPOCHS_DIR" --out-root "$REPS_DIR" \
        --representations neurolm --neurolm-dir "$NEUROLM_DIR" --neurolm-checkpoint "$NEUROLM_CHECKPOINT" --device $DEVICE

## Stage 3 — fine-tune CBraMod to recognise the word (about 1–2 h on an A100)

The whole pretrained model learns, on single-reader epochs, to pick the read word's embedding among the words in the batch; 5 folds split by sentence give predictions for every word from a model that never saw its sentence. The control run is identical but with the words shuffled across training epochs. The table at the end shows how often the read word is ranked first among the held-out vocabulary: real vs shuffled.

In [ ]:
!python scripts/finetune_eeg_encoder.py --epochs-dir "$EPOCHS_DIR" --out-root "$REPS_DIR" --device $DEVICE
OUT_3 = f'{RESULTS}/finetuning'
os.makedirs(OUT_3, exist_ok=True)
!cp "$REPS_DIR"/cbramod_ft_training.* "$OUT_3/"
push('finetuning', OUT_3)

## Stage 4 — the same tests for every representation (about 5–15 min each)

`decode_eeg_to_text.py`: 2-vs-2 (all pairs and matched pairs), retrieval, stacked on word features, sentiment of the decoded text. `word_information.py`: embedding prediction, sentence identification, word properties. Then one comparison table.

In [ ]:
OUT_4 = f'{RESULTS}/tests'
ORDER = ['eye_tracking', 'trt', 'frp', 'raw', 'cbramod', 'neurolm', 'cbramod_ft', 'cbramod_ft_shuffled']
for rep in ORDER:
    rep_dir = f'{REPS_DIR}/{rep}'
    if not os.path.isdir(rep_dir):
        print('not built:', rep); continue
    if not os.path.exists(f'{OUT_4}/decoding_{rep}.json'):
        !python scripts/decode_eeg_to_text.py --dataset zuco --word-eeg-dir "$rep_dir" --tag $rep --out-dir "$OUT_4" --device $DEVICE
    if not os.path.exists(f'{OUT_4}/word_info_{rep}.json'):
        !python scripts/word_information.py --word-eeg-dir "$rep_dir" --tag $rep --out-dir "$OUT_4" --device $DEVICE
!cp "$REPS_DIR"/cbramod_ft_training.md "$OUT_4/" 2>/dev/null
!python scripts/summarize_word_information.py --results-dir "$OUT_4" --order {' '.join(ORDER)}
push('tests', OUT_4)

## Stage 5 (optional) — EEG-to-text from the best representation

Runs the v4 EEG-to-text pipeline (ridge map to mBART embeddings, positive control, test-time swaps) on one representation, e.g. the fine-tuned CBraMod. English only (9 readers, sentiment-task sentences).

In [ ]:
BEST = 'cbramod_ft'
OUT_5 = f'{RESULTS}/eeg_to_text'
!python scripts/run_eeg_to_text.py --zuco-word-eeg-dir "{REPS_DIR}/{BEST}" --results-dir "$OUT_5" --run-tag $BEST \
    --settings en --encoders continuous --inputs eeg noise shuffled_eeg word_vectors --device $DEVICE
push('eeg_to_text', f'{OUT_5}/{BEST}')

## Summary

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT_4}/word_information_summary.md').read()))